# Adım 6 — Olasılık kaydı + çoklu tohum (BERT / RoBERTa, leave-one-dataset-out)

**Amaç (05'in devamı):**
1. Her örnek için modelin **olasılığını kaydetmek**, böylece "masumlarda olasılık ile metin uzunluğu ilişkisi var mı?" sorusuna bakabilmek.
2. Aynı koşuları **tohum 43 ve 44** ile tekrarlayıp sonuçların rastgeleliğe ne kadar duyarlı olduğunu görmek (tohum 42 zaten 05'te yapıldı).

Protokol 05 ile **aynı** (Karar 8). Veri bölme sabit kalır (eşik seti her seferinde aynı), sadece **eğitimin rastgeleliği** (parti sırası, yeni katman başlangıcı) tohuma göre değişir.

**Nasıl çalıştırılır:**
1. Üst menüden `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`.
2. Hücreleri **sırayla** çalıştır. **"A) Olasılık kaydı"** hücresini çalıştır (2 koşu, ~10 dk), sonra **"B) Olasılık-uzunluk analizi"** hücresini çalıştır. Çıktıyı Claude'a yapıştır.
3. Sonra **"C) Tohum 43 ve 44"** hücresini çalıştır (16 koşu, tahmini ~90 dk). Colab kopsa bile yeniden çalıştırınca kaldığı yerden devam eder. Bitince **"D) Tohumlar arası özet"** hücresini çalıştır.
Hata çıkarsa hatayı olduğu gibi Claude'a yapıştır.

**Önemli:** Bu defterdeki **yeni** parçalar (olasılık kaydı, tohum parametresi, analiz ve özet hücreleri) henüz gerçek veriyle ve GPU ile **denenmedi**. Mantık sahte veriyle ve minik bir modelle test edildi. Geri kalanı 05'te gerçek veriyle çalıştı.

## 1) Kütüphaneler

In [ ]:
!pip -q install transformers scikit-learn

## 2) Gerekli parçalar ve ayarlar
`TOHUM` her koşuda aynı rastgelelik için sabit (02 ile aynı: 42). Aşağıdaki ayar değerleri Karar 8'den geliyor.

In [ ]:
import os, time, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupShuffleSplit

# ---- Ayarlar (Karar 8) ----
TOHUM = 42            # her koşuda aynı rastgelelik
TUR = 3               # veriyi kaç kez baştan sona gezecek
PARTI = 16            # aynı anda kaç örnek işlenecek
OGRENME_HIZI = 2e-5   # her adımda ne kadar düzeltme (küçük tutulur)
MAKS_UZUNLUK = 128    # uzun metinler bu kadar parçada kesilir (bildiride sınırlılık olarak yazılacak)
HEDEF_FPR = 0.01      # %1 yanlış pozitif oranı

cihaz = "cuda" if torch.cuda.is_available() else "cpu"
KARISIK = (cihaz == "cuda")   # GPU'da hızlandırmak için yarı hassasiyet (fp16)
print("Kullanılan cihaz:", cihaz)
if cihaz == "cpu":
    print("UYARI: GPU yok! Çalışma zamanı türünden T4 GPU seçmelisin, yoksa eğitim çok yavaş olur.")
else:
    print("Kart:", torch.cuda.get_device_name(0))


def tohum_ayarla(t=TOHUM):
    random.seed(t); np.random.seed(t); torch.manual_seed(t)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(t)

## 3) Google Drive'ı bağla (sonuçları kaybetmemek için)
Sonuç tablosu her koşudan sonra **senin** Drive'ındaki `YiitliProje_sonuclar` klasörüne yazılır. Colab kopsa bile kaybolmaz. Bağlanırken izin penceresi çıkacak, izin ver.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SONUC_KLASORU = "/content/drive/MyDrive/YiitliProje_sonuclar"
os.makedirs(SONUC_KLASORU, exist_ok=True)
SONUC_DOSYASI = SONUC_KLASORU + "/sonuclar_ahmet_v2.csv"   # her koşunun ölçümleri (tohum sütunlu)
HATA_DOSYASI = SONUC_KLASORU + "/hatalar_ahmet_v2.csv"     # modelin yanıldığı örnekler
OLASILIK_DOSYASI = SONUC_KLASORU + "/olasiliklar_ahmet.csv" # HER örnek için model olasılığı
print("Sonuçlar şuraya yazılacak:", SONUC_DOSYASI)

## 4) veri_birlesik.csv'yi indir ve kontrol et
Dosya Yiğit'in Drive'ında. Aşağıdaki kod Google hesabınla Drive'dan dosyayı **dosya numarasıyla** indirir (Yiğit seni klasöre eklediği için erişimin var). Kontrol: **23.081 satır x 7 sütun** çıkmalı.

In [ ]:
from google.colab import auth
auth.authenticate_user()                       # Google hesabınla giriş
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

servis = build("drive", "v3")

def drive_indir(dosya_id, yerel_ad):
    """Drive'daki bir dosyayı numarasıyla Colab'a indirir."""
    istek = servis.files().get_media(fileId=dosya_id)
    with open(yerel_ad, "wb") as f:
        indirici = MediaIoBaseDownload(f, istek)
        bitti = False
        while not bitti:
            _, bitti = indirici.next_chunk()
    print("İndirildi:", yerel_ad, "| boyut (bayt):", os.path.getsize(yerel_ad))

drive_indir("BURAYA_DRIVE_DOSYA_NUMARASI", "veri_birlesik.csv")   # veri_birlesik.csv

In [ ]:
df = pd.read_csv("veri_birlesik.csv")
print("Şekil:", df.shape, "(beklenen: 23081 x 7)")
print("Sütunlar:", list(df.columns))
print()
print(df.groupby(["rol", "kaynak", "bolum"]).agg(satir=("label", "size"), zararli=("label", "sum")))

# Basit güvenlik kontrolleri: beklediğimiz değerler yoksa burada dururuz
beklenen = {"safeguard", "neuralchemy", "wambosec", "jackhhao", "deepset"}
assert beklenen <= set(df["kaynak"]), f"Beklenen kaynak adları yok! Bulunan: {set(df['kaynak'])}"
assert set(df["bolum"]) <= {"train", "val", "test"}, f"Beklenmeyen bölüm adı: {set(df['bolum'])}"
assert set(df["label"]) <= {0, 1}, "label sadece 0 ve 1 olmalı"
print("\nBoş metin sayısı:", df["text"].isna().sum())
df = df.dropna(subset=["text"]).reset_index(drop=True)
df["text"] = df["text"].astype(str)

## 5) Veriyi koşuya göre bölen fonksiyon (Karar 8)
- **Eğitim havuzu** = çıkarılmayan 2 ana veri setinin `train` + `val` bölümleri.
- Havuzun **%10'u** "eşik seçme seti" olarak ayrılır (modele gösterilmez). Sadece "%1 yanlış pozitifte hangi eşik?" sorusu için kullanılır.
- **Test** = çıkarılan veri setinin **tamamı** (model hiç görmedi) + `jackhhao (test)` + `deepset (tamamı)`.
- `kaynak_cikarilan=None` verilirse **aynı dağılımda referans** koşusu: 3 ana veri setiyle eğit, 3'ünün `test` bölümünde ölç (genelleme sayılmaz, sadece kıyas için).

neuralchemy'de aynı metnin çoğaltılmış hâlleri `grup` sütunuyla işaretli. %10'luk ayırmayı **gruba göre** yapıyoruz ki aynı grubun parçaları hem eğitime hem eşik setine düşüp eşiği olduğundan iyi göstermesin.

In [ ]:
def egitim_ayir(havuz):
    """Havuzu %90 eğitim / %10 eşik-seçme setine böler (grup bütünlüğünü korur)."""
    havuz = havuz.reset_index(drop=True)
    # grup boşsa her satır kendi başına bir grup sayılır
    gruplar = [f"tek_{i}" if pd.isna(g) else str(g) for i, g in enumerate(havuz["grup"])]
    ayirici = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=TOHUM)
    egitim_idx, esik_idx = next(ayirici.split(havuz, groups=gruplar))
    return havuz.iloc[egitim_idx].reset_index(drop=True), havuz.iloc[esik_idx].reset_index(drop=True)


def kosu_verisi(df, kaynak_cikarilan):
    ana = df[df["rol"] == "ana"]
    dis = df[df["rol"] == "dis_test"]
    if kaynak_cikarilan is None:
        havuz = ana[ana["bolum"].isin(["train", "val"])]
        testler = {}
        for k in sorted(ana["kaynak"].unique()):
            testler[f"{k} (test)"] = ana[(ana["kaynak"] == k) & (ana["bolum"] == "test")]
        testler["ana_uc_test (birlesik)"] = ana[ana["bolum"] == "test"]
    else:
        havuz = ana[(ana["kaynak"] != kaynak_cikarilan) & (ana["bolum"].isin(["train", "val"]))]
        testler = {f"{kaynak_cikarilan} (tamami)": ana[ana["kaynak"] == kaynak_cikarilan]}
    testler["jackhhao (test)"] = dis[(dis["kaynak"] == "jackhhao") & (dis["bolum"] == "test")]
    testler["deepset (tamami)"] = dis[dis["kaynak"] == "deepset"]
    egitim, esik_seti = egitim_ayir(havuz)
    return egitim, esik_seti, testler

## 6) Ölçüt fonksiyonları
Model her metin için "zararlı olma olasılığı" (0 ile 1 arası) üretir. Karar vermek için bir **eşik** gerekir: olasılık eşikten büyükse "zararlı".

- **Varsayılan eşik 0,5:** accuracy, precision, recall, F1 bununla hesaplanır.
- **ROC-AUC:** Eşikten bağımsız genel ayırma gücü (1,0 mükemmel, 0,5 şans).
- **TPR@%1 FPR (a), iyimser:** Test setinin kendi ROC eğrisinde, masumların en fazla %1'ini yanlış engelleyen en iyi eşikte yakalama oranı. Eşiği test setine bakarak seçtiği için iyimserdir.
- **TPR@%1 FPR (b), gerçekçi:** Eşiği **eşik seçme setinde** (modelin görmediği, eğitim dağılımından) seçip **sabit** tutarız. Görülmemiş testte gerçek yakalama oranını (TPR) ve gerçek yanlış pozitif oranını (FPR) raporlarız. Yeni dağılımda FPR %1'den büyük çıkabilir, bu beklenen ve önemli bir bulgudur.

In [ ]:
def esik_sec(y_esik, p_esik, hedef_fpr=HEDEF_FPR):
    """Masumların (label=0) en fazla hedef_fpr kadarı eşiğin üstünde kalacak şekilde eşik seç.
    Karar kuralı: olasılık > eşik ise 'zararlı'."""
    masum = np.sort(p_esik[y_esik == 0])
    k = int(np.floor(hedef_fpr * len(masum)))   # izin verilen yanlış engelleme sayısı
    return float(masum[len(masum) - 1 - k])


def tpr_roc(y, p, hedef_fpr=HEDEF_FPR):
    """Test setinin kendi ROC eğrisinden: FPR <= hedef iken ulaşılan en yüksek TPR."""
    if len(set(y)) < 2:
        return float("nan")
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr[fpr <= hedef_fpr]))


def olc(y, p, esik_sabit):
    y = np.asarray(y); p = np.asarray(p)
    tah = (p >= 0.5).astype(int)                    # varsayılan eşik 0,5
    tn = int(((tah == 0) & (y == 0)).sum()); fp = int(((tah == 1) & (y == 0)).sum())
    fn = int(((tah == 0) & (y == 1)).sum()); tp = int(((tah == 1) & (y == 1)).sum())
    iki_sinif = len(set(y)) == 2
    tah_s = (p > esik_sabit).astype(int)            # eğitim tarafında seçilmiş sabit eşik
    masum_say = int((y == 0).sum()); zararli_say = int((y == 1).sum())
    return {
        "n": len(y), "n_zararli": zararli_say, "n_masum": masum_say,
        "dogruluk": accuracy_score(y, tah),
        "kesinlik": precision_score(y, tah, zero_division=0),
        "duyarlilik": recall_score(y, tah, zero_division=0),
        "f1": f1_score(y, tah, zero_division=0),
        "roc_auc": roc_auc_score(y, p) if iki_sinif else float("nan"),
        "TP": tp, "FN": fn, "FP": fp, "TN": tn,
        "tpr_roc_1fpr_(a)": tpr_roc(y, p),
        "esik_sabit": esik_sabit,
        "tpr_sabit_(b)": float(((tah_s == 1) & (y == 1)).sum() / max(zararli_say, 1)),
        "fpr_sabit_(b)": float(((tah_s == 1) & (y == 0)).sum() / max(masum_say, 1)),
    }

## 7) Model yükleme, eğitim ve tahmin fonksiyonları
- `tokenizer` = metni sayılara çeviren "kelime parçalayıcı".
- Eğitim, 02'deki gibi: her turda tüm örnekleri 16'lık partilerle gezer, hatayı (loss) hesaplar, modeli düzeltir.
- GPU'da yarı hassasiyet (fp16) kullanılır, 02'den tek fark budur. Sadece hız içindir.

In [ ]:
def modeli_yukle(model_adi):
    tokenizer = AutoTokenizer.from_pretrained(model_adi)
    model = AutoModelForSequenceClassification.from_pretrained(model_adi, num_labels=2)
    return tokenizer, model.to(cihaz)


def kodla(tokenizer, metinler):
    k = tokenizer(list(metinler), padding="max_length", truncation=True,
                  max_length=MAKS_UZUNLUK, return_tensors="pt")
    return k["input_ids"], k["attention_mask"]


def egit(model, tokenizer, egitim_df, tohum=TOHUM):
    ids, maske = kodla(tokenizer, egitim_df["text"])
    etiket = torch.tensor(egitim_df["label"].astype(int).values)
    uretec = torch.Generator().manual_seed(tohum)   # parti sırası tohuma göre değişir
    yukleyici = DataLoader(TensorDataset(ids, maske, etiket), batch_size=PARTI,
                           shuffle=True, generator=uretec)
    optimizer = torch.optim.AdamW(model.parameters(), lr=OGRENME_HIZI)
    olcekleyici = torch.amp.GradScaler("cuda", enabled=KARISIK)
    for tur in range(TUR):
        model.train(); toplam = 0.0
        for i, m, e in yukleyici:
            i, m, e = i.to(cihaz), m.to(cihaz), e.to(cihaz)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=KARISIK):
                cikti = model(input_ids=i, attention_mask=m, labels=e)   # tahmin + hata
            optimizer.zero_grad()
            olcekleyici.scale(cikti.loss).backward()   # hatanın nereden geldiğini hesapla
            olcekleyici.step(optimizer)                # modeli düzelt
            olcekleyici.update()
            toplam += cikti.loss.item()
        print(f"  Tur {tur+1}/{TUR} | ortalama loss: {toplam/len(yukleyici):.4f}")


def olasilik_tahmin(model, tokenizer, metinler, parti=64):
    """Her metin için 'zararlı olma olasılığını' (0-1) döndürür. Eğitim yapmaz."""
    model.eval()
    ids, maske = kodla(tokenizer, metinler)
    sonuc = []
    with torch.no_grad():
        for b in range(0, len(ids), parti):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=KARISIK):
                mantik = model(input_ids=ids[b:b+parti].to(cihaz),
                               attention_mask=maske[b:b+parti].to(cihaz)).logits
            sonuc.append(torch.softmax(mantik.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(sonuc)

## 8) Bir koşuyu baştan sona yapan fonksiyon
Bir koşu = (model, çıkarılan veri seti). Adımlar: veriyi böl → modeli yükle → eğit → eşik seç → her test setinde ölç → sonuçları CSV'ye ekle.

In [ ]:
def kosu(model_adi, kaynak_cikarilan, tohum=TOHUM):
    ad = f"{model_adi} | cikarilan={kaynak_cikarilan or 'YOK (ayni dagilim referansi)'} | tohum={tohum}"
    print("=" * 70); print("KOŞU:", ad)
    t0 = time.time()
    tohum_ayarla(tohum)   # eğitimin rastgeleliği. Veri bölme (egitim_ayir) hep TOHUM=42 ile, sabit.
    egitim, esik_seti, testler = kosu_verisi(df, kaynak_cikarilan)
    print(f"Eğitim: {len(egitim)} (zararlı oranı {egitim['label'].mean():.3f}) | "
          f"Eşik seti: {len(esik_seti)} (masum: {(esik_seti['label']==0).sum()})")
    if (esik_seti["label"] == 0).sum() < 200:
        print("  UYARI: eşik setinde 200'den az masum var, %1 FPR eşiği kaba bir tahmindir.")

    tokenizer, model = modeli_yukle(model_adi)
    egit(model, tokenizer, egitim, tohum)

    p_esik = olasilik_tahmin(model, tokenizer, esik_seti["text"])
    esik = esik_sec(esik_seti["label"].values, p_esik)
    print(f"Eşik (eşik setinde masumların %{HEDEF_FPR*100:.0f}'ini engelleyen): {esik:.6f}")

    satirlar, hatalar, olasiliklar = [], [], []
    for test_adi, t in testler.items():
        p = olasilik_tahmin(model, tokenizer, t["text"])
        r = olc(t["label"].values, p, esik)
        r.update({"model": model_adi, "cikarilan": kaynak_cikarilan or "YOK", "tohum": tohum, "test": test_adi})
        # HER örnek için olasılığı kaydet (metnin kendisi değil, sadece uzunluk ve satır sayısı)
        olasiliklar.append(pd.DataFrame({
            "model": model_adi, "cikarilan": kaynak_cikarilan or "YOK", "tohum": tohum, "test": test_adi,
            "etiket": t["label"].values, "olasilik": p, "kategori": t["kategori"].values,
            "uzunluk": t["text"].str.len().values, "satir_sayisi": t["text"].str.count("\n").values}))
        satirlar.append(r)
        yanlis = np.where((p >= 0.5).astype(int) != t["label"].values)[0]
        for j in yanlis:   # modelin yanıldığı örnekler (ilk 300 karakter)
            hatalar.append({"model": model_adi, "cikarilan": kaynak_cikarilan or "YOK", "test": test_adi,
                            "tohum": tohum, "gercek_etiket": int(t["label"].values[j]), "olasilik": float(p[j]),
                            "kategori": t["kategori"].values[j], "text": t["text"].values[j][:300]})
    sure = (time.time() - t0) / 60
    tablo = pd.DataFrame(satirlar)
    tablo["sure_dk"] = round(sure, 1)
    sutunlar = ["model", "cikarilan", "tohum", "test", "n", "n_zararli", "n_masum", "dogruluk", "kesinlik",
                "duyarlilik", "f1", "roc_auc", "TP", "FN", "FP", "TN", "tpr_roc_1fpr_(a)",
                "esik_sabit", "tpr_sabit_(b)", "fpr_sabit_(b)", "sure_dk"]
    tablo = tablo[sutunlar]

    # Her koşudan sonra Drive'a ekle (Colab kopsa bile kaybolmasın)
    tablo.to_csv(SONUC_DOSYASI, mode="a", header=not os.path.exists(SONUC_DOSYASI), index=False)
    pd.concat(olasiliklar).to_csv(OLASILIK_DOSYASI, mode="a", header=not os.path.exists(OLASILIK_DOSYASI), index=False)
    if hatalar:
        pd.DataFrame(hatalar).to_csv(HATA_DOSYASI, mode="a", header=not os.path.exists(HATA_DOSYASI), index=False)

    # GPU belleğini boşalt (sonraki koşu için)
    del model, tokenizer
    if cihaz == "cuda":
        torch.cuda.empty_cache()
    print(f"Koşu süresi: {sure:.1f} dk")
    return tablo

## 9) A) Olasılık kaydı (tohum 42, safeguard çıkarılmış, BERT ve RoBERTa)
Bu iki koşu 05'tekilerin **aynısı** (tohum 42), ama bu sefer her örnek için olasılık da kaydedilir. Yaklaşık 10 dakika.

In [ ]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
for model_adi in ["bert-base-uncased", "roberta-base"]:
    kosu(model_adi, "safeguard", tohum=42)
pd.read_csv(SONUC_DOSYASI).round(4)

## 10) B) Olasılık-uzunluk analizi (sadece masum örnekler, safeguard çıkarılmış koşu)
Soru: Model, **masum** metinlerde uzun metne daha yüksek "zararlı" olasılığı veriyor mu?
- **Spearman korelasyonu:** +1 = uzunluk arttıkça olasılık hep artıyor, 0 = ilişki yok, -1 = tersi.
- **Uzunluk dilimi tablosu:** Masumları uzunluğa göre 4 eşit gruba böler (en kısa dörtte bir ... en uzun dörtte bir). Her grupta ortalama olasılığı ve yanlış alarm oranını (olasılık >= 0,5) verir.

In [ ]:
from scipy.stats import spearmanr
o = pd.read_csv(OLASILIK_DOSYASI)
m = o[(o["cikarilan"] == "safeguard") & (o["test"] == "safeguard (tamami)") & (o["etiket"] == 0)]
for (model_adi, tohum), g in m.groupby(["model", "tohum"]):
    rho, _ = spearmanr(g["uzunluk"], g["olasilik"])
    print(f"{model_adi} | tohum {tohum} | masum n={len(g)} | Spearman(uzunluk, olasılık) = {rho:.3f}")
    g = g.assign(dilim=pd.qcut(g["uzunluk"], 4, labels=["1 (en kısa)", "2", "3", "4 (en uzun)"], duplicates="drop"))
    ozet = g.groupby("dilim", observed=True).agg(ortanca_uzunluk=("uzunluk", "median"),
                                                  ort_olasilik=("olasilik", "mean"),
                                                  yanlis_alarm=("olasilik", lambda x: (x >= 0.5).mean()))
    print(ozet.round(3)); print()

## 11) C) Tohum 43 ve 44 (tüm koşular)
2 model x 4 koşu x 2 tohum = 16 koşu, tahmini ~90 dk (tahmin, 05'te koşu başına 4-7 dk sürdü). Bitmiş koşuları atlar, Colab kopsa bile yeniden çalıştırınca kaldığı yerden devam eder. Her koşu sonucu hemen Drive'a yazılır.

In [ ]:
MODELLER = ["bert-base-uncased", "roberta-base"]
CIKARILANLAR = ["safeguard", "neuralchemy", "wambosec", None]   # None = aynı dağılım referansı
TOHUMLAR = [43, 44]

def bitti_mi(model_adi, cikarilan, tohum):
    if not os.path.exists(SONUC_DOSYASI):
        return False
    s = pd.read_csv(SONUC_DOSYASI)
    return ((s["model"] == model_adi) & (s["cikarilan"] == (cikarilan or "YOK")) & (s["tohum"] == tohum)).any()

for tohum in TOHUMLAR:
    for model_adi in MODELLER:
        for cikarilan in CIKARILANLAR:
            if bitti_mi(model_adi, cikarilan, tohum):
                print("ATLANDI (zaten bitmiş):", model_adi, cikarilan, tohum)
                continue
            kosu(model_adi, cikarilan, tohum=tohum)
print("Bitti.")

## 12) D) Tohumlar arası özet
Tohum 42'nin sonuçları 05'ten (Drive'da `sonuclar_ahmet_temiz.csv`) alınır, 43 ve 44 bu defterden. Her (model, çıkarılan set, test seti) için **ortalama ± standart sapma** ve tohum sayısı. Sadece 3 tohumu da olan satırlar gösterilir. Standart sapma 3 değerden hesaplanır, kaba bir ölçüdür.

In [ ]:
drive_indir("BURAYA_DRIVE_DOSYA_NUMARASI", "sonuclar_tohum42.csv")   # 05'in temiz sonuçları (tohum 42)
eski = pd.read_csv("sonuclar_tohum42.csv"); eski["tohum"] = 42
yeni = pd.read_csv(SONUC_DOSYASI)
tum = pd.concat([eski, yeni]).drop_duplicates(subset=["model", "cikarilan", "test", "tohum"])

metrikler = ["dogruluk", "duyarlilik", "roc_auc", "tpr_roc_1fpr_(a)", "tpr_sabit_(b)", "fpr_sabit_(b)"]
gruplar = tum.groupby(["model", "cikarilan", "test"])
ozet = gruplar[metrikler].agg(["mean", "std"])
ozet.columns = [f"{a}_{b}" for a, b in ozet.columns]
ozet["n_tohum"] = gruplar["tohum"].nunique()
ozet = ozet[ozet["n_tohum"] == 3].round(3)
ozet.to_csv(SONUC_KLASORU + "/tohum_ozet_ahmet.csv")
ozet